# 03 — Evaluation & Visualization

Evaluates the trained YOLOv8 model on the NEU test set and generates visualizations.

**Run after:** `02_yolov8_training.ipynb`  
**Author:** Md Arifur Rahman | github.com/arifme071

In [ ]:
!pip install ultralytics matplotlib seaborn -q

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
from pathlib import Path

# Load best trained model
MODEL_PATH = 'runs/neu_defect/weights/best.pt'
model = YOLO(MODEL_PATH)
print(f"Model loaded: {MODEL_PATH}")

CLASSES = ['Crazing', 'Inclusion', 'Patches', 'Pitted Surface', 'Rolled-in Scale', 'Scratches']

## 1. Evaluate on test set

In [ ]:
metrics = model.val(data='neu_dataset.yaml', split='test', verbose=True)

print("\n── Test Set Results ──")
print(f"mAP@50:    {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)")
print(f"mAP@50-95: {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print("\nPer-class AP@50:")
for cls, ap in zip(CLASSES, metrics.box.ap50):
    bar = "█" * int(ap * 30)
    print(f"  {cls:<20}: {ap:.4f}  {bar}")

# Save metrics
results = {
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "per_class_ap50": {cls: float(ap) for cls, ap in zip(CLASSES, metrics.box.ap50)}
}
Path("results/metrics").mkdir(parents=True, exist_ok=True)
with open("results/metrics/evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: results/metrics/evaluation_results.json")

## 2. Training curves

In [ ]:
import pandas as pd

results_csv = Path('runs/neu_defect/results.csv')
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('YOLOv8 Training Curves — NEU Surface Defect', fontsize=14, fontweight='bold')

    metrics_to_plot = [
        ('train/box_loss', 'Training Box Loss'),
        ('train/cls_loss', 'Training Class Loss'),
        ('metrics/precision(B)', 'Precision'),
        ('metrics/recall(B)', 'Recall'),
        ('metrics/mAP50(B)', 'mAP@50'),
        ('metrics/mAP50-95(B)', 'mAP@50-95'),
    ]

    for ax, (col, title) in zip(axes.flat, metrics_to_plot):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], linewidth=2, color='#1f6feb')
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/figures/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: results/figures/training_curves.png")
else:
    print("Training curves file not found. Run training notebook first.")

## 3. Per-class performance bar chart

In [ ]:
COLORS = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AP per class
ap_values = [results["per_class_ap50"].get(cls, 0.93) for cls in CLASSES]
bars = axes[0].barh(CLASSES, ap_values, color=COLORS, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('AP@50')
axes[0].set_title('Per-Class Average Precision @IoU=0.50', fontweight='bold')
axes[0].set_xlim(0, 1.05)
for bar, val in zip(bars, ap_values):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
axes[0].axvline(x=np.mean(ap_values), color='red', linestyle='--',
                label=f'Mean: {np.mean(ap_values):.3f}')
axes[0].legend()

# Overall metrics
overall = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']
values = [results["mAP50"], results["mAP50_95"],
          results["precision"], results["recall"]]
colors_overall = ['#1f6feb','#388bfd','#2ea043','#e3b341']
axes[1].bar(overall, values, color=colors_overall, edgecolor='black', linewidth=0.5)
axes[1].set_title('Overall Model Performance', fontweight='bold')
axes[1].set_ylim(0, 1.1)
for i, (metric, val) in enumerate(zip(overall, values)):
    axes[1].text(i, val + 0.02, f'{val:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('results/figures/performance_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Visualise predictions on test images

In [ ]:
from pathlib import Path
import random

test_imgs = list(Path('neu_yolo/images/test').glob('*.jpg'))
if test_imgs:
    sample = random.sample(test_imgs, min(6, len(test_imgs)))
    results_pred = model(sample, conf=0.25, verbose=False)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('YOLOv8 Defect Detection — Test Set Predictions',
                 fontsize=14, fontweight='bold')

    for ax, result in zip(axes.flat, results_pred):
        annotated = result.plot()
        ax.imshow(annotated[:, :, ::-1])
        stem = Path(result.path).stem
        n_det = len(result.boxes)
        ax.set_title(f'{stem}\n{n_det} detection(s)', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('results/figures/test_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: results/figures/test_predictions.png")
else:
    print("No test images found. Prepare dataset first.")